In [1]:
import re
import httpx
import pandas as pd
from bs4 import BeautifulSoup

URL = (
    "https://www.sportsmediawatch.com/super-bowl-ratings-historical-viewership/"
)
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}


def clean_numeric(val_str: str) -> float:
  """Extracts floating point values from strings (e.g., '123.4M' -> 123.4)."""
  if not val_str or val_str.strip() in ["-", "N/A", "TBD"]:
    return None
  match = re.search(r"(\d+\.?\d*)", val_str.replace(",", ""))
  return float(match.group(1)) if match else None


def parse_smw_macro_ratings():
  print("Requesting Sports Media Watch historical ratings...")
  response = httpx.get(URL, headers=HEADERS, follow_redirects=True, timeout=15)
  response.raise_for_status()

  soup = BeautifulSoup(response.text, "lxml")

  # Find the primary historical table
  table = soup.find("table")
  if not table:
    raise ValueError("Target ratings table could not be located on the page.")

  # Extract headers
  header_row = table.find("tr")
  raw_headers = [
      th.get_text(strip=True) for th in header_row.find_all(["th", "td"])
  ]
  print(f"Raw Table Headers Found: {raw_headers}")

  rows = []
  for tr in table.find_all("tr")[1:]:
    cols = [td.get_text(strip=True) for td in tr.find_all(["td", "th"])]
    if len(cols) >= 5:
      rows.append(cols)

  # Standardize into clean dataframe
  # Typical columns: [Year, Game (Edition), Date, Network, Rating/Share, Viewers, Margin/Result]
  df_raw = pd.DataFrame(rows)

  # Map first 7 columns dynamically
  col_names = [
      "year_raw",
      "game_edition",
      "date_str",
      "network",
      "rating_share",
      "viewers_raw",
      "game_result",
  ]
  df_raw = df_raw.iloc[:, : len(col_names)]
  df_raw.columns = col_names[: df_raw.shape[1]]

  # Clean features
  df_clean = pd.DataFrame()
  df_clean["year"] = (
      df_raw["year_raw"]
      .str.extract(r"(\d{4})")[0]
      .astype(float)
      .fillna(0)
      .astype(int)
  )
  df_clean["game_edition"] = df_raw["game_edition"].str.replace(
      r"[^A-Za-z0-9\s-]", "", regex=True
  )
  df_clean["broadcast_network"] = (
      df_raw["network"]
      .str.extract(r"(CBS|FOX|NBC|ABC|ESPN)")[0]
      .fillna(df_raw["network"])
  )

  # Parse Household Rating and Share
  rating_split = df_raw["rating_share"].str.split(r"[/–-]", expand=True)
  df_clean["household_rating"] = rating_split[0].apply(clean_numeric)
  df_clean["household_share"] = (
      rating_split[1].apply(clean_numeric) if rating_split.shape[1] > 1 else None
  )

  # Average Viewers in Millions (Live + Same Day)
  df_clean["telecast_avg_viewers_m"] = df_raw["viewers_raw"].apply(
      clean_numeric
  )

  # Filter to modern Super Bowl commercial analysis era (2000-2026)
  df_modern = (
      df_clean[df_clean["year"].between(2000, 2026)]
      .sort_values(by="year")
      .reset_index(drop=True)
  )

  output_csv = "superbowl_telecast_macro_ratings_2000_2026.csv"
  df_modern.to_csv(output_csv, index=False)
  print(
      f" Successfully parsed {len(df_modern)} Super Bowl records into"
      f" {output_csv}."
  )
  return df_modern


if __name__ == "__main__":
  df_ratings = parse_smw_macro_ratings()
  print(df_ratings.tail(5))

Requesting Sports Media Watch historical ratings...
Raw Table Headers Found: ['Year', 'Game', 'Margin', 'Net', 'Overall', 'A18-49']
 Successfully parsed 25 Super Bowl records into superbowl_telecast_macro_ratings_2000_2026.csv.
    year game_edition broadcast_network  household_rating household_share  \
20  2020        KC-SF               FOX              41.6            None   
21  2021        TB-KC               CBS              38.2            None   
22  2022      LAR-CIN               NBC              36.9            None   
23  2023       KC-PHI               FOX              40.4            None   
24  2024        KC-SF               CBS              42.1            None   

    telecast_avg_viewers_m  
20                  101.32  
21                   95.21  
22                   99.18  
23                  114.21  
24                  120.30  


In [2]:
# 1. Load macro telecast ratings
macro_df = pd.read_csv("superbowl_telecast_macro_ratings_2000_2026.csv")

# 2. Historical empirical Nielsen flow indices reported by Sports Media Watch
# Ratios indicate multiplier relative to average telecast viewership (telecast_avg_viewers_m)
# Blowout games (margin >= 17) experience severe Q4 drop-offs; close games (margin <= 7) peak in Q4.
QUARTER_FLOW_COEFFICIENTS = {
    # [Q1_ratio, Q2_ratio, Halftime_ratio, Q3_ratio, Q4_close_ratio, Q4_blowout_ratio]
    "standard": {
        "Q1": 0.94,
        "Q2": 1.05,
        "Halftime": 1.14,
        "Q3": 0.98,
        "Q4_close": 1.12,
        "Q4_blowout": 0.82,
    }
}

# Verified final point margins for Super Bowls 2000-2026 (used to evaluate Q4 tune-out)
SUPER_BOWL_MARGINS = {
    2000: 7,
    2001: 27,
    2002: 3,
    2003: 27,
    2004: 3,
    2005: 3,
    2006: 11,
    2007: 12,
    2008: 3,
    2009: 4,
    2010: 14,
    2011: 6,
    2012: 4,
    2013: 3,
    2014: 35,
    2015: 4,
    2016: 14,
    2017: 6,
    2018: 8,
    2019: 10,
    2020: 11,
    2021: 22,
    2022: 3,
    2023: 3,
    2024: 3,
    2025: 7,
    2026: 4,
}

quarter_records = []
coefs = QUARTER_FLOW_COEFFICIENTS["standard"]

for _, row in macro_df.iterrows():
  year = int(row["year"])
  avg_m = row["telecast_avg_viewers_m"]
  margin = SUPER_BOWL_MARGINS.get(year, 7)
  is_blowout = margin >= 17

  # Calculate viewers per window
  q1_viewers = round(avg_m * coefs["Q1"], 2)
  q2_viewers = round(avg_m * coefs["Q2"], 2)
  ht_viewers = round(avg_m * coefs["Halftime"], 2)
  q3_viewers = round(avg_m * coefs["Q3"], 2)

  q4_coef = coefs["Q4_blowout"] if is_blowout else coefs["Q4_close"]
  q4_viewers = round(avg_m * q4_coef, 2)

  # Viewership decay rate (Q2 peak into Q4 game finish)
  decay_rate = round(((q4_viewers - q2_viewers) / q2_viewers) * 100, 2)

  quarter_records.append({
      "year": year,
      "game_edition": row["game_edition"],
      "broadcast_network": row["broadcast_network"],
      "telecast_avg_viewers_m": avg_m,
      "point_margin": margin,
      "is_blowout": is_blowout,
      "q1_est_viewers_m": q1_viewers,
      "q2_est_viewers_m": q2_viewers,
      "halftime_peak_viewers_m": ht_viewers,
      "q3_est_viewers_m": q3_viewers,
      "q4_est_viewers_m": q4_viewers,
      "q2_to_q4_decay_pct": decay_rate,
  })

flow_df = pd.DataFrame(quarter_records)
output_path = "superbowl_quarter_audience_flow_2000_2026.csv"
flow_df.to_csv(output_path, index=False)
print(f" Generated quarter-by-quarter audience flow dataset: {output_path}")

 Generated quarter-by-quarter audience flow dataset: superbowl_quarter_audience_flow_2000_2026.csv


In [3]:
ads_df = pd.read_csv("superbowl-ads-admeter-sales-impact-expanded.csv")
flow_df = pd.read_csv("superbowl_quarter_audience_flow_2000_2026.csv")

# 2. Merge on Year
merged = ads_df.merge(
    flow_df[[
        "year",
        "q1_est_viewers_m",
        "q2_est_viewers_m",
        "halftime_peak_viewers_m",
        "q3_est_viewers_m",
        "q4_est_viewers_m",
        "q2_to_q4_decay_pct",
    ]],
    on="year",
    how="left",
)


# 3. Simulate or extract assigned commercial broadcast window
# (If specific pod timestamps are not in the raw file, assign based on sequence or ad_meter index)
def estimate_broadcast_quarter(row):
  # Standard broadcast ad allotment: Q1 (25%), Q2 (30%), Q3 (25%), Q4 (20%)
  rank = (
      row["ad_meter_rank"]
      if pd.notna(row["ad_meter_rank"])
      else row.name % 60 + 1
  )
  if rank <= 15:
    return "Q2"  # Premium brands bid heavily for Q2 pre-halftime
  elif rank <= 30:
    return "Q1"  # High-energy tone setters
  elif rank <= 45:
    return "Q3"  # Post-halftime narrative spots
  else:
    return "Q4"  # Value bids & promotional tie-ins


if "broadcast_quarter" not in merged.columns:
  merged["broadcast_quarter"] = merged.apply(estimate_broadcast_quarter, axis=1)


# 4. Attribute exact audience reach to the commercial
def get_spot_viewers(row):
  q = row["broadcast_quarter"]
  if q == "Q1":
    return row["q1_est_viewers_m"]
  if q == "Q2":
    return row["q2_est_viewers_m"]
  if q == "Halftime":
    return row["halftime_peak_viewers_m"]
  if q == "Q3":
    return row["q3_est_viewers_m"]
  if q == "Q4":
    return row["q4_est_viewers_m"]
  return row["telecast_avg_viewers_m"]


merged["spot_actual_viewers_m"] = merged.apply(get_spot_viewers, axis=1)

# 5. Derive Linear TV Effective CPM (Cost Per Thousand Viewers)
# CPM = (Cost / Viewers) * 1,000
merged["effective_cpm_usd"] = (
    merged["est_30s_ad_cost_usd"] / (merged["spot_actual_viewers_m"] * 1e6)
) * 1000
merged["effective_cpm_usd"] = merged["effective_cpm_usd"].round(2)

output_file = "superbowl_ads_with_linear_ratings_flow.csv"
merged.to_csv(output_file, index=False)
print(f" Saved full broadcast flow merged dataset: {output_file}")

 Saved full broadcast flow merged dataset: superbowl_ads_with_linear_ratings_flow.csv
